In [58]:
using Random
using RobustNeuralNetworks
using Flux
using CUDA
using FileIO
using Images
using ImageMagick
using ImageTransformations  # For image resizing


In [59]:
# Random seed for consistency
rng = MersenneTwister(42)
print
# Model specification
nu = 64 * 64 * 3  # Number of inputs (size of flattened image)
output_size = 2             # Number of outputs (human vs non-human classification)
nh =fill(64,4)             # Two hidden layers with 128 and 64 neurons respectively
hidden_size=64

64

In [60]:
init = Flux.glorot_normal(rng)
initb(n) = Flux.glorot_normal(rng, n)
model = Chain(
    Dense(nu, hidden_size, Flux.relu, init=init, bias=initb(hidden_size)),
    Dense(hidden_size, hidden_size, Flux.relu, init=init, bias=initb(hidden_size)),
    Dense(hidden_size, hidden_size, Flux.relu, init=init, bias=initb(hidden_size)),
    Dense(hidden_size, hidden_size, Flux.relu, init=init, bias=initb(hidden_size)),
    Dense(hidden_size, output_size, init=init, bias=initb(output_size)),Flux.softmax


)|>gpu

Chain(
  Dense(12288 => 64, relu),             # 786_496 parameters
  Dense(64 => 64, relu),                # 4_160 parameters
  Dense(64 => 64, relu),                # 4_160 parameters
  Dense(64 => 64, relu),                # 4_160 parameters
  Dense(64 => 2),                       # 130 parameters
  NNlib.softmax,
)                   # Total: 10 arrays, 799_106 parameters, 1.562 KiB.

In [62]:
function load_images_from_folder(folder_path)
    images = []
    supported_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".gif"]
    for file in readdir(folder_path)
        ext = lowercase(splitext(file)[2])
        if ext in supported_extensions
            img_path = joinpath(folder_path, file)
            img = load(img_path)
            push!(images, img)
        end
    end
    return images
end

load_images_from_folder (generic function with 1 method)

In [63]:
function preprocess_images(images, target_size=(64, 64))
    return [reshape(channelview(imresize(img, target_size)), :) ./ 255.0 for img in images]
end


preprocess_images (generic function with 2 methods)

In [64]:

# Paths to image folders
human_images_path = "C:\\Users\\jonat\\OneDrive\\Desktop\\Human_vs_non_human\\human-and-non-human\\versions\\1\\human-and-non-human\\training_set\\training_set\\humans"
non_human_images_path = "C:\\Users\\jonat\\OneDrive\\Desktop\\Human_vs_non_human\\human-and-non-human\\versions\\1\\human-and-non-human\\training_set\\training_set\\non-humans"


"C:\\Users\\jonat\\OneDrive\\Desktop\\Human_vs_non_human\\human-and-non-human\\versions\\1\\human-and-non-human\\training_set\\training_set\\non-humans"

In [65]:
# Load and preprocess images
human_images = preprocess_images(load_images_from_folder(human_images_path))
non_human_images = preprocess_images(load_images_from_folder(non_human_images_path))

# Prepare labels
human_labels = [1 for _ in 1:length(human_images)]
non_human_labels = [0 for _ in 1:length(non_human_images)]


4006-element Vector{Int64}:
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 ⋮
 0
 0
 0
 0
 0
 0
 0
 0
 0

In [66]:

# Combine images and labels
images = vcat(human_images, non_human_images)
labels = vcat(human_labels, non_human_labels)

# Convert labels to one-hot encoded vectors
y_train = Flux.onehotbatch(labels, 0:1)


2×8017 OneHotMatrix(::Vector{UInt32}) with eltype Bool:
 ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  …  1  1  1  1  1  1  1  1  1  1  1  1
 1  1  1  1  1  1  1  1  1  1  1  1  1     ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅

In [67]:
X_train = cat(images..., dims=2) |> gpu  # Stack images along second dimension
Y_train = y_train |> gpu

2×8017 OneHotMatrix(::CuArray{UInt32, 1, CUDA.DeviceMemory}) with eltype Bool:
 ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  …  1  1  1  1  1  1  1  1  1  1  1  1
 1  1  1  1  1  1  1  1  1  1  1  1  1     ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅

In [68]:
Y_train = y_train |> gpu
# Prepare data for training
data = [(X_train, Y_train)]

1-element Vector{Tuple{CuArray{Float32, 2, CUDA.DeviceMemory}, OneHotMatrix{UInt32, CuArray{UInt32, 1, CUDA.DeviceMemory}}}}:
 ([0.0 0.0 … 0.002675894 0.0032602844; 0.0 0.0 … 0.0026143792 0.0032602844; … ; 0.002522107 0.0014148405 … 0.0013840831 0.0038754325; 0.0018300654 0.0015993848 … 0.0014302192 0.0038600538], Bool[0 0 … 1 1; 1 1 … 0 0])

In [69]:
using Flux.Optimisers: Adam
using Flux.Losses: logitcrossentropy

In [70]:

optimizer = Adam()
loss(model,x,y) = Flux.logitcrossentropy(model(x), y)


loss (generic function with 1 method)

In [71]:
using Statistics
using Flux:OneHotMatrix
# Check test accuracy during training
compare(y::OneHotMatrix, ŷ) = maximum(ŷ, dims=1) .== maximum(y.*ŷ, dims=1)
accuracy(model, x, y::OneHotMatrix) = mean(compare(y, model(x)))

# Callback function to show results while training
function progress(model, iter,a,b)
    train_loss = round(loss(model, Flux.flatten(a), b), digits=4)
    #test_acc = round(accuracy(model, x_test, y_test), digits=4)
    @show iter train_loss #test_acc
    println()
end


progress (generic function with 1 method)

In [72]:
function train_model(data, epochs,opt,loss1,mod)
    for epoch in 1:epochs
        Flux.train!(loss1, model, data, opt)|>gpu
        println("Epoch $epoch completed")
        #progress(model)
        progress(model,epoch,X_train,Y_train)
    end
end

train_model (generic function with 1 method)

In [73]:
using BSON



opt_state = Flux.setup(Adam(0.058), model)
train_model(data, 100,opt_state,loss,model)  # Train for 10 epochs

bson("non_robust_test1.bson", Dict("model" => model |> gpu))


Epoch 1 completed
iter = 1
train_loss = 0.8127f0

Epoch 2 completed
iter = 2
train_loss = 0.8008f0

Epoch 3 completed
iter = 3
train_loss = 0.7686f0

Epoch 4 completed
iter = 4
train_loss = 0.7486f0

Epoch 5 completed
iter = 5
train_loss = 0.7276f0

Epoch 6 completed
iter = 6
train_loss = 0.6375f0

Epoch 7 completed
iter = 7
train_loss = 0.5994f0

Epoch 8 completed
iter = 8
train_loss = 0.5625f0

Epoch 9 completed
iter = 9
train_loss = 0.5538f0

Epoch 10 completed
iter = 10
train_loss = 0.5541f0

Epoch 11 completed
iter = 11
train_loss = 0.5359f0

Epoch 12 completed
iter = 12
train_loss = 0.5344f0

Epoch 13 completed
iter = 13
train_loss = 0.5442f0

Epoch 14 completed
iter = 14
train_loss = 0.5059f0

Epoch 15 completed
iter = 15
train_loss = 0.53f0

Epoch 16 completed
iter = 16
train_loss = 0.489f0

Epoch 17 completed
iter = 17
train_loss = 0.504f0

Epoch 18 completed
iter = 18
train_loss = 0.4738f0

Epoch 19 completed
iter = 19
train_loss = 0.4868f0

Epoch 20 completed
iter = 20
train

In [75]:
train_acc = accuracy(model, X_train, Y_train)*100


96.60720967943121

In [76]:
human_images_path1 ="C:\\Users\\jonat\\OneDrive\\Desktop\\Human_vs_non_human\\human-and-non-human\\versions\\1\\human-and-non-human\\test_set\\test_set\\humans"
non_human_images_path1 = "C:\\Users\\jonat\\OneDrive\\Desktop\\Human_vs_non_human\\human-and-non-human\\versions\\1\\human-and-non-human\\test_set\\test_set\\non-humans"

# Load and preprocess images
human_images1 = preprocess_images(load_images_from_folder(human_images_path1))
non_human_images1 = preprocess_images(load_images_from_folder(non_human_images_path1))


1309-element Vector{Vector{Float64}}:
 [0.002306805074971165, 0.002522106881968474, 0.0025528642829680892, 0.002091503267973856, 0.002291426374471357, 0.002399077277970012, 0.0018762014609765476, 0.002106881968473664, 0.0023375624759707806, 0.0020761245674740486  …  0.0031680123029604, 0.003552479815455594, 0.003721645520953479, 0.003552479815455594, 0.0030142252979623225, 0.0032449058054594385, 0.0034602076124567475, 0.00010765090349865437, 0.0002921953094963476, 0.001199538638985006]
 [0.001968473663975394, 0.0014917339484813532, 0.0012764321414840446, 0.0008765859284890427, 0.0007381776239907728, 0.0008458285274894272, 0.002183775470972703, 0.0019992310649750095, 0.0021530180699730873, 0.0028143021914648213  …  0.001138023836985775, 0.0008612072279892349, 0.0006766628219915417, 0.0009688581314878893, 0.0007996924259900039, 0.0006151480199923107, 0.0009073433294886583, 0.0007843137254901962, 0.0006151480199923107, 0.0008919646289888505]
 [0.0015071126489811613, 0.0014609765474817378,

In [77]:

# Prepare labels
human_labels1 = [1 for _ in 1:length(human_images1)]
non_human_labels1 = [0 for _ in 1:length(non_human_images1)]


# Combine images and labels
images1 = vcat(human_images1, non_human_images1)
labels1 = vcat(human_labels1, non_human_labels1)
X_test = cat(images1..., dims=2) |> gpu  # Stack images along second dimension
y_test = Flux.onehotbatch(labels1, 0:1)

Y_test = y_test|> gpu




2×2723 OneHotMatrix(::CuArray{UInt32, 1, CUDA.DeviceMemory}) with eltype Bool:
 ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  …  1  1  1  1  1  1  1  1  1  1  1  1
 1  1  1  1  1  1  1  1  1  1  1  1  1     ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅

In [78]:
test_acc = accuracy(model, X_test, Y_test)*100


95.3360264414249